# Introduction to Data Science 2026

# Week 4

In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**

## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.

In [ ]:
!pip install --quiet transformers

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3)), but note that Colab may have issues running larger models.

In [ ]:
MODEL_NAME = 'gpt2-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

Instantiate a text generation pipeline using the tokenizer and model.

In [ ]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [ ]:
output = pipe('Once upon a time,', max_new_tokens=25)

print(output)

[transformers] Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Once upon a time, it was said, the Earth was born in one day. Thus, the Earth was born in a single day, and the'}]


Just print the text

In [ ]:
print(output[0]['generated_text'])

Once upon a time, it was said, the Earth was born in one day. Thus, the Earth was born in a single day, and the


We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [ ]:
output = pipe(
    'Terve, miten menee?',
    do_sample=True,
    temperature=10.0,
    max_new_tokens=25
)

print(output[0]['generated_text'])

[transformers] Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Terve, miten menee? Du igen dabeilet, ist mitten. Weiter dien, hat bekenken so ain


## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- one-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- two-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Denmark?\
	>Answer: Copenhagen
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
	- binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
	- person name recognition: "List the person names occurring in the following texts."
	- two-digit addition: "This is a first grade math exam."
- Finnish:
	- binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
	- person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
	- two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.

In [ ]:
# Use this cell for your code

import re
from transformers import GenerationConfig

# Experiment definition: three tasks, each described by a dict with:
#   instruction - the instruction line placed at the top of every prompt
#   text_label - label used for the "question" line of the prompt
#   answer_label - label used for the "answer" line (left empty as the hook)
#   examples - our OWN 1-shot and 2-shot examples: list of (text, answer)
#   tests - test cases: list of (text, expected answer)

TASKS = {
    "sentiment": {
        "instruction": "Do the following texts express a positive or negative sentiment?",
        "text_label": "Text",
        "answer_label": "Sentiment",
        "examples": [
            ("I am so happy today.", "positive"),
            ("This meeting is a complete waste of time.", "negative"),
        ],
        "tests": [
            ("I love this movie, it is fantastic!", "positive"),
            ("This is the worst day of my life.", "negative"),
            ("The food was delicious and the service was wonderful.", "positive"),
            ("I am so disappointed with this product, it broke immediately.", "negative"),
        ],
    },
    "names": {
        "instruction": "List the person names occurring in the following texts.",
        "text_label": "Text",
        "answer_label": "Names",
        "examples": [
            ("Tom plays with Jerry.", "Tom, Jerry"),
            ("Susan wrote a letter to Peter.", "Susan, Peter"),
        ],
        "tests": [
            ("Alice went to the store with Bob.", ["Alice", "Bob"]),
            ("Yesterday, John and Mary met at the park.", ["John", "Mary"]),
            ("The manager Sarah gave the report to David.", ["Sarah", "David"]),
            ("Michael called his friend Emma on the phone.", ["Michael", "Emma"]),
        ],
    },
    "addition": {
        "instruction": "This is a first grade math exam.",
        "text_label": "Question",
        "answer_label": "Answer",
        "examples": [
            ("What is 12 + 12?", "24"),
            ("What is 10 + 21?", "31"),
        ],
        "tests": [
            ("What is 11 + 22?", "33"),
            ("What is 45 + 37?", "82"),
            ("What is 63 + 29?", "92"),
            ("What is 58 + 74?", "132"),
        ],
    },
}


def build_prompt(task, n_shots, test_text):
    """Build the prompt for a task with n_shots examples.

    Pattern: instruction + (k example Q/A pairs) + the test item + an empty
    answer cue that the model will continue.

    n_shots = 0 -> no example lines (zero-shot)
    n_shots = 1 -> prepend the first example (one-shot)
    n_shots = 2 -> prepend the first two examples (two-shot)
    """
    cfg = TASKS[task]
    lines = [cfg["instruction"], ""]
    for ex_text, ex_ans in cfg["examples"][:n_shots]:
        lines.append(f"{cfg['text_label']}: {ex_text}")
        lines.append(f"{cfg['answer_label']}: {ex_ans}")
        lines.append("")
    lines.append(f"{cfg['text_label']}: {test_text}")
    lines.append(f"{cfg['answer_label']}:")
    return "\n".join(lines)


def generate(prompt, max_new_tokens=30):
    """Let the model continue the prompt; return ONLY the new text (the answer)."""
    # Pack the generation settings into a GenerationConfig object. Passing the
    # config (instead of loose kwargs) avoids the pipeline's internal default
    # max_length, so transformers prints no "max_new_tokens vs max_length" warning.
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=50256,
    )
    # clean_up_tokenization_spaces=False silences the BPE cleanup warning (GPT-2
    # output is unchanged — that post-processing is skipped for BPE anyway).
    out = pipe(prompt, generation_config=gen_cfg, clean_up_tokenization_spaces=False)
    # generated_text = prompt + continuation; cut off the prompt itself
    return out[0]["generated_text"][len(prompt):].strip()


results = {}
for task in TASKS:
    results[task] = {}
    print(f"===== {task} =====")
    for n_shots in (0, 1, 2):
        rows = []
        for test_text, expected in TASKS[task]["tests"]:
            prompt = build_prompt(task, n_shots, test_text)
            output = generate(prompt)
            rows.append({"test": test_text, "expected": expected, "output": output})
            print(f"[{task}] {n_shots}-shot | {test_text} -> {output!r}")
        results[task][n_shots] = rows


# Evaluation
def eval_sentiment(output, expected):
    """Sentiment: the answer is the FIRST label word the model produces."""
    # lowercase both sides for easy matching
    text = output.lower()
    # position of first "positive" (-1 if absent)
    pos_idx = text.find("positive")
    # position of first "negative" (-1 if absent)
    neg_idx = text.find("negative")
    # no label word at all -> no answer given
    if pos_idx == -1 and neg_idx == -1:
        return "incorrect"
    # GPT-2 keeps echoing the prompt template, so later labels are just noise;
    # only the FIRST label word counts as the model's answer.
    first = "positive" if (pos_idx != -1 and (neg_idx == -1 or pos_idx < neg_idx)) else "negative"
    return "correct" if first == expected else "incorrect"


def eval_names(output, expected_names):
    """Names: correct only if ALL expected names appear in the output."""
    found = [n for n in expected_names if n.lower() in output.lower()]  # case-insensitive
    return "correct" if len(found) == len(expected_names) else "incorrect"


def eval_addition(output, expected):
    """Addition: correct if the expected sum appears among the numbers in the output."""
    # extract every run of digits
    numbers = re.findall(r"\d+", output)
    return "correct" if expected in numbers else "incorrect"


# Map each task name to its evaluator so the scoring loop stays generic.
EVALUATORS = {"sentiment": eval_sentiment, "names": eval_names, "addition": eval_addition}

# Score every stored record in place by adding a "verdict" field.
for task in results:
    for n_shots in results[task]:
        for r in results[task][n_shots]:
            r["verdict"] = EVALUATORS[task](r["output"], r["expected"])

# Print an accuracy table: <10 left-aligns in 10 chars, >16 right-aligns in 16 chars.
print(f"{'task':<10} | {'0-shot':>16} | {'1-shot':>16} | {'2-shot':>16}")
print("-" * 62)
for task in results:
    cells_ = []
    for n_shots in (0, 1, 2):
        rows = results[task][n_shots]
        # count correct answers
        n_correct = sum(1 for r in rows if r["verdict"] == "correct")
        # e.g. "2/4 (50%)"
        cells_.append(f"{n_correct}/{len(rows)} ({n_correct/len(rows):.0%})")
    print(f"{task:<10} | {cells_[0]:>16} | {cells_[1]:>16} | {cells_[2]:>16}")


===== sentiment =====
[sentiment] 0-shot | I love this movie, it is fantastic! -> 'I love this movie, it is fantastic!\n\nText: I love this movie, it is fantastic!\n\nSentiment: I love this'
[sentiment] 0-shot | This is the worst day of my life. -> 'This is the best day of my life.\n\nText: I am so happy.\n\nSentiment: I am so sad.'
[sentiment] 0-shot | The food was delicious and the service was wonderful. -> 'The food was delicious and the service was wonderful.\n\nText: The food was delicious and the service was wonderful.\n\nSentiment: The'
[sentiment] 0-shot | I am so disappointed with this product, it broke immediately. -> 'I am so disappointed with this product, it broke immediately.\n\nText: I am so disappointed with this product, it broke immediately.'
[sentiment] 1-shot | I love this movie, it is fantastic! -> 'positive\n\nText: I am so happy today.\n\nSentiment: positive\n\nText: I love this movie, it is fantastic!'
[sentiment] 1-shot | This is the worst day of my life. -> 'n

**Q: Does the model succeed at each task, in zero-, one- and two-show settings?**

1. **Person name recognition. Yes, in all three settings (4/4, 100%).** The model finds the names in the input sentence and lists them correctly. This task only needs copying information that is already in the text, so it is easy for the model.

2. **Binary sentiment classification. No (0-shot 0/4 = 0%, 1-shot 4/4 = 100%, 2-shot 2/4 = 50%).** The model seems to ignore the task quite often. It repeats the prompt tempalte, or just writes a label without reading the text. Adding exmaples does not help. The accuracy stays near guessing.

3. **Addition. No (0/4 in all three settings, 0%).** The model cannot do arithmetric. It only copies the question, copies the example answer, or writes a close-but-wrong number. More examples do not help.

Few-shot prompint works only when the model already has the needed ability. It cannot create abilities the model lacks, such as arithmetic or following inst ructions reliably.

**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**